In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets

data = pd.concat([X, y], axis=1)

target_col = "quality"

data = data.replace("?", np.nan)
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

numeric_cols = data.columns

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

data[target_col] = data[target_col].astype(int)

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


In [4]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)



================ SINGLE RUN ================
Training TabDDPM...
[0]
17
{'num_classes': 6, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(17)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.344 Sum: 0.344
Step 1000/1000 MLoss: 0.0 GLoss: 0.3232 Sum: 0.3232
mlp
Sample timestep    0
Discrete cols: []
Num shape:  (1000, 11)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 368.75it/s]|
Column Shapes Score: 76.08%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 292.85it/s]|
Column Pair Trends Score: 73.9%

Overall Score (Average): 74.99%

TabDDPM: 0.7499


In [5]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 193.84it/s]|
Column Shapes Score: 85.74%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 68.43it/s]|
Column Pair Trends Score: 82.12%

Overall Score (Average): 83.93%

ForestDiffusion: 0.8393


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=42),
    'SVC-RBF': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=2000, random_state=42),
    'GaussianNB': GaussianNB(),
}

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in train_df.columns:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        train_df[col] = train_df[col].fillna(train_df[col].median())

    for col in test_df.columns:
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
        test_df[col] = test_df[col].fillna(train_df[col].median())

    train_df[label_col] = train_df[label_col].round().clip(0, 10).astype(int)
    test_df[label_col] = test_df[label_col].round().clip(0, 10).astype(int)

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            stratify_train = (
                y_train_full
                if y_train_full.nunique() > 1 and y_train_full.value_counts().min() >= 2
                else None
            )

            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_train
            )

            stratify_test = (
                y_test_full
                if y_test_full.nunique() > 1 and y_test_full.value_counts().min() >= 2
                else None
            )

            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_test
            )

            if y_train_split.nunique() < 2:
                continue

            scaler = StandardScaler()

            X_train_s = scaler.fit_transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            try:
                clf.fit(X_train_s, y_train_split)

                y_pred = clf.predict(X_test_s)

                accuracy_scores.append(accuracy_score(y_test_split, y_pred))

                f1_scores.append(
                    f1_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                precision_scores.append(
                    precision_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                recall_scores.append(
                    recall_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

            except Exception:
                continue

        if len(accuracy_scores) == 0:
            results.append({
                "Model": name,
                "Accuracy Mean": np.nan,
                "Accuracy Std": np.nan,
                "F1 Mean": np.nan,
                "F1 Std": np.nan,
                "Precision Mean": np.nan,
                "Precision Std": np.nan,
                "Recall Mean": np.nan,
                "Recall Std": np.nan,
                "Accuracy ± SD": "N/A",
                "F1 ± SD": "N/A",
                "Precision ± SD": "N/A",
                "Recall ± SD": "N/A"
            })
            continue

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1) if len(accuracy_scores) > 1 else 0

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1) if len(f1_scores) > 1 else 0

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1) if len(precision_scores) > 1 else 0

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1) if len(recall_scores) > 1 else 0

        results.append({
            "Model": name,
            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,
            "Accuracy ± SD": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 ± SD": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision ± SD": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall ± SD": f"{rec_mean:.4f} ± {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False,
        na_position="last"
    )


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        scaler = StandardScaler()

        X_train_real = scaler.fit_transform(X_train_real)
        X_test_real = scaler.transform(X_test_real)

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)


--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: quality
Target classes:
quality
6    458
5    302
7    167
4     36
8     32
3      5
Name: count, dtype: int64
Running LogisticRegression...
Running SVC-RBF...
Running KNN...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running MLP...
Running GaussianNB...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
5,ExtraTrees,0.5445 ± 0.0362,0.5181 ± 0.0348,0.5235 ± 0.0334,0.5445 ± 0.0362
6,GradientBoost,0.5285 ± 0.0293,0.5114 ± 0.0297,0.5115 ± 0.0396,0.5285 ± 0.0293
4,RandomForest,0.5270 ± 0.0206,0.5022 ± 0.0183,0.5017 ± 0.0175,0.5270 ± 0.0206
1,SVC-RBF,0.5255 ± 0.0331,0.4765 ± 0.0340,0.5045 ± 0.0426,0.5255 ± 0.0331
0,LogisticRegression,0.5160 ± 0.0323,0.4856 ± 0.0344,0.4923 ± 0.0388,0.5160 ± 0.0323
7,MLP,0.4935 ± 0.0217,0.4887 ± 0.0210,0.4871 ± 0.0202,0.4935 ± 0.0217
2,KNN,0.4905 ± 0.0266,0.4727 ± 0.0254,0.4749 ± 0.0350,0.4905 ± 0.0266
3,DecisionTree,0.4460 ± 0.0503,0.4469 ± 0.0484,0.4514 ± 0.0476,0.4460 ± 0.0503
8,GaussianNB,0.4115 ± 0.0513,0.4141 ± 0.0505,0.4373 ± 0.0505,0.4115 ± 0.0513


In [12]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "ForestDiffusion",
    "TabDDPM"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label_col="quality",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy ± SD",
            "F1 ± SD",
            "Precision ± SD",
            "Recall ± SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    if label_col not in synthetic_train_df.columns:
        print(f"{label_col} not found in {synth_name}. Skipping.")
        continue

    synthetic_train_df[label_col] = pd.to_numeric(
        synthetic_train_df[label_col],
        errors="coerce"
    )

    synthetic_train_df[label_col] = (
        synthetic_train_df[label_col]
        .fillna(real_data[label_col].mode()[0])
        .round()
        .astype(int)
    )

    synthetic_train_df = synthetic_train_df.dropna()

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label_col="quality",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy ± SD",
                "F1 ± SD",
                "Precision ± SD",
                "Recall ± SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy ± SD_TRTR",
                "Accuracy ± SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby(
        "Synthetic_Model",
        as_index=False
    )[
        [
            "Accuracy_Drop",
            "F1_Drop",
            "Precision_Drop",
            "Recall_Drop"
        ]
    ]
    .mean()
    .sort_values(
        "Accuracy_Drop",
        ascending=True
    )
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)

TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,ExtraTrees,0.5445 ± 0.0362,0.5181 ± 0.0348,0.5235 ± 0.0334,0.5445 ± 0.0362
6,GradientBoost,0.5285 ± 0.0293,0.5114 ± 0.0297,0.5115 ± 0.0396,0.5285 ± 0.0293
4,RandomForest,0.5270 ± 0.0206,0.5022 ± 0.0183,0.5017 ± 0.0175,0.5270 ± 0.0206
1,SVC-RBF,0.5255 ± 0.0331,0.4765 ± 0.0340,0.5045 ± 0.0426,0.5255 ± 0.0331
0,LogisticRegression,0.5160 ± 0.0323,0.4856 ± 0.0344,0.4923 ± 0.0388,0.5160 ± 0.0323
7,MLP,0.4935 ± 0.0217,0.4887 ± 0.0210,0.4871 ± 0.0202,0.4935 ± 0.0217
2,KNN,0.4905 ± 0.0266,0.4727 ± 0.0254,0.4749 ± 0.0350,0.4905 ± 0.0266
3,DecisionTree,0.4460 ± 0.0503,0.4469 ± 0.0484,0.4514 ± 0.0476,0.4460 ± 0.0503
8,GaussianNB,0.4115 ± 0.0513,0.4141 ± 0.0505,0.4373 ± 0.0505,0.4115 ± 0.0513


CTGAN not found in synthetic_datasets. Skipping.
CopulaGAN not found in synthetic_datasets. Skipping.
TVAE not found in synthetic_datasets. Skipping.
GaussianCopula not found in synthetic_datasets. Skipping.
ForestDiffusion - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,ExtraTrees,0.6725 ± 0.0699,0.6653 ± 0.0709,0.6784 ± 0.0778,0.6725 ± 0.0699
4,RandomForest,0.6525 ± 0.0646,0.6452 ± 0.0642,0.6582 ± 0.0683,0.6525 ± 0.0646
6,GradientBoost,0.6110 ± 0.0559,0.6083 ± 0.0558,0.6145 ± 0.0570,0.6110 ± 0.0559
7,MLP,0.5990 ± 0.0703,0.5983 ± 0.0673,0.6021 ± 0.0645,0.5990 ± 0.0703
2,KNN,0.5880 ± 0.0484,0.5809 ± 0.0484,0.5793 ± 0.0497,0.5880 ± 0.0484
1,SVC-RBF,0.5870 ± 0.0334,0.5692 ± 0.0348,0.5867 ± 0.0453,0.5870 ± 0.0334
3,DecisionTree,0.5425 ± 0.0441,0.5459 ± 0.0447,0.5549 ± 0.0467,0.5425 ± 0.0441
0,LogisticRegression,0.5255 ± 0.0390,0.5152 ± 0.0410,0.5172 ± 0.0470,0.5255 ± 0.0390
8,GaussianNB,0.4130 ± 0.0378,0.4114 ± 0.0415,0.4497 ± 0.0516,0.4130 ± 0.0378


ForestDiffusion - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogisticRegression,0.4695 ± 0.0215,0.3508 ± 0.0367,0.3639 ± 0.0401,0.4695 ± 0.0215
1,SVC-RBF,0.4600 ± 0.0033,0.2938 ± 0.0109,0.2592 ± 0.1057,0.4600 ± 0.0033
5,ExtraTrees,0.4425 ± 0.0300,0.3757 ± 0.0279,0.3408 ± 0.0348,0.4425 ± 0.0300
4,RandomForest,0.4370 ± 0.0216,0.3664 ± 0.0225,0.3262 ± 0.0205,0.4370 ± 0.0216
6,GradientBoost,0.4025 ± 0.0326,0.3625 ± 0.0263,0.3505 ± 0.0342,0.4025 ± 0.0326
2,KNN,0.3980 ± 0.0333,0.3645 ± 0.0296,0.3653 ± 0.0527,0.3980 ± 0.0333
7,MLP,0.3880 ± 0.0341,0.3683 ± 0.0280,0.3551 ± 0.0261,0.3880 ± 0.0341
3,DecisionTree,0.3125 ± 0.0353,0.3106 ± 0.0357,0.3110 ± 0.0376,0.3125 ± 0.0353
8,GaussianNB,0.2590 ± 0.0443,0.2461 ± 0.0501,0.3157 ± 0.0509,0.2590 ± 0.0443


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,TabDDPM,ExtraTrees,0.1020,0.142403,0.182727,0.1020,0.5445 ± 0.0362,0.4425 ± 0.0300
1,TabDDPM,GradientBoost,0.1260,0.148989,0.160967,0.1260,0.5285 ± 0.0293,0.4025 ± 0.0326
2,TabDDPM,RandomForest,0.0900,0.135843,0.175566,0.0900,0.5270 ± 0.0206,0.4370 ± 0.0216
3,TabDDPM,SVC-RBF,0.0655,0.182625,0.245393,0.0655,0.5255 ± 0.0331,0.4600 ± 0.0033
4,TabDDPM,LogisticRegression,0.0465,0.134874,0.128371,0.0465,0.5160 ± 0.0323,0.4695 ± 0.0215
5,TabDDPM,MLP,0.1055,0.120408,0.131933,0.1055,0.4935 ± 0.0217,0.3880 ± 0.0341
6,TabDDPM,KNN,0.0925,0.108270,0.109565,0.0925,0.4905 ± 0.0266,0.3980 ± 0.0333
7,TabDDPM,DecisionTree,0.1335,0.136286,0.140418,0.1335,0.4460 ± 0.0503,0.3125 ± 0.0353
8,TabDDPM,GaussianNB,0.1525,0.167904,0.121544,0.1525,0.4115 ± 0.0513,0.2590 ± 0.0443


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,-0.078667,-0.091503,-0.095197,-0.078667
1,TabDDPM,0.101556,0.141956,0.155165,0.101556


In [13]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
